In [1]:
pip install fastmcp

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 1.0 MB/s  0:00:0055.5 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.7 MB/s  0:00:00m 1.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━ 21/29 [openapi-pydantic]  WARNING: The script keyring is installed in '/Users/apoorv.apoorv/Library/Python/3.14/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 24/29 [mcp]  WARNING: The script mcp is installed in '/Users/apoorv.apoorv/Library/Python/3.14/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 25/29 [fastmcp-slim]  WARNING: The script fastmcp is installed in '/Users/apoorv.apoorv/Library/Python/3

In [3]:
pip install nest-asyncio

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install langchain-mcp-adapters

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install langchain-google-genai

Defaulting to user installation because normal site-packages is not writeable
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-google-genai]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
import asyncio
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Optional for later (keep commented)
# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY,
)

In [9]:
llm.invoke("Hi")

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f0893-8332-7a73-9e21-2b2419a93480-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 47, 'total_tokens': 49, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 37}})

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [16]:
from agent_tools import (
    fetch_country_info,
    search_cheap_flights,
    search_specific_flights,
    find_hotels,
    get_coordinates,
    get_weather,
    get_driving_distance,
)

In [17]:
get_driving_distance

StructuredTool(name='get_driving_distance', description='Calculates the driving distance and duration between two GPS points.\n\nDEPENDENCIES:\nYou CANNOT call this with city names. You MUST run \'get_coordinates\' first\nto get the (lat, lon) for both Start and End points.\n\nARGS:\n    lat1, lon1: Start point coordinates (float).\n    lat2, lon2: End point coordinates (float).\n\nRETURNS:\n    str: A string describing the route.\n         Format: "{Distance} km, {Time} mins"', args_schema=<class 'langchain_core.utils.pydantic.get_driving_distance'>, func=<function get_driving_distance at 0x116c226c0>)

In [10]:
# import asyncio
# import nest_asyncio
# nest_asyncio.apply()

# from langchain_mcp_adapters.client import MultiServerMCPClient
# client = MultiServerMCPClient(
#     {
#         "hotel_booking": {
#             "transport": "http",
#             "url": "http://localhost:8000/sse",
#         }
#     }
# )

# mcp_tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

# print("Discovered tools:", [t.name for t in mcp_tools])

In [22]:
import asyncio
from mcp.client.sse import sse_client
from mcp import ClientSession

async def list_my_tools():
    url = "http://localhost:8000/sse"
    print(f"Connecting to {url}...")
    
    try:
        # Open the SSE connection
        async with sse_client(url) as streams:
            async with ClientSession(streams[0], streams[1]) as session:
                # The required handshake
                await session.initialize()
                
                # Ask for the tools
                response = await session.list_tools()
                return response
    except Exception as e:
        print(f"❌ Error: {e}")

In [23]:
tool_list = asyncio.run(list_my_tools())

Connecting to http://localhost:8000/sse...
❌ Error: Not currently running on any asynchronous event loop. Available async backends: asyncio, trio


In [25]:
t = tool_list.tools[0]
t.inputSchema
# getattr(t,"args_schema")

AttributeError: 'NoneType' object has no attribute 'tools'

In [47]:
t.name

'book_hotel_room'

In [13]:
import asyncio
import nest_asyncio
from mcp.client.sse import sse_client
from mcp import ClientSession

In [24]:
nest_asyncio.apply()
async def book_hotel():
    async with sse_client("http://localhost:8000/sse") as streams:
        async with ClientSession(streams[0], streams[1]) as session:
            await session.initialize()
            result = await session.call_tool("book_hotel_room", {"hotel_name": "Taj Convention", "location":"Goa", "date":"2026-06-22"})
            return result

In [55]:
asyncio.get_event_loop().run_until_complete(book_hotel())

CallToolResult(meta=None, content=[TextContent(type='text', text="SUCCESS: Hotel 'Taj Convention' in Goa has been booked for 2026-06-22. Reservation ID: 93D83731. Status: CONFIRMED", annotations=None, meta=None)], structuredContent={'result': "SUCCESS: Hotel 'Taj Convention' in Goa has been booked for 2026-06-22. Reservation ID: 93D83731. Status: CONFIRMED"}, isError=False)

### LOAD THE TOOLS WITH LANGCHAIN

In [58]:
from langchain_mcp_adapters.tools import load_mcp_tools
async with sse_client("http://localhost:8000/sse") as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        
        # 2. Let LangChain auto-wrap everything instantly!
        print("Auto-loading tools from MCP server...")
        mcp_tools = await load_mcp_tools(session)

Auto-loading tools from MCP server...


In [59]:
mcp_tools

[StructuredTool(name='book_hotel_room', description='\n    Books a reservation at a specific hotel.\n\n    Args:\n        hotel_name: The name of the hotel (e.g. "Taj Mahal Palace")\n        location: The city or location (e.g. "Mumbai")\n        date: The check-in date in YYYY-MM-DD format.\n\n    Returns:\n        A confirmation string with a reservation ID.\n    ', args_schema={'properties': {'hotel_name': {'title': 'Hotel Name', 'type': 'string'}, 'location': {'title': 'Location', 'type': 'string'}, 'date': {'title': 'Date', 'type': 'string'}}, 'required': ['hotel_name', 'location', 'date'], 'title': 'book_hotel_roomArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000022660EE6D40>)]

In [60]:
fetch_country_info

StructuredTool(name='fetch_country_info', description='Fetches broad details about a country.\n\nWHEN TO USE:\nUse this when the user asks generic questions about a destination\'s region, currency, or language.\n\nARGS:\n    country_name (str): The full English name of the country (e.g., "France", "India").\n\nRETURNS:\n    dict: A dictionary with the following keys:\n          - \'country\': (str) Common name of the country.\n          - \'region\': (str) Geographic region (e.g., \'Europe\').\n          - \'currencies\': (str) A string listing currency names and codes.\n          - \'coordinates\': (list) [latitude, longitude] of the country center.', args_schema=<class 'langchain_core.utils.pydantic.fetch_country_info'>, func=<function fetch_country_info at 0x0000022659A0EA70>)

### Calling with Langraph

In [56]:
from agent_tools import (
    fetch_country_info,
    search_cheap_flights,
    search_specific_flights,
    find_hotels,
    get_coordinates,
    get_weather,
    get_driving_distance,
)

internal_tools = [
    fetch_country_info,
    search_cheap_flights,
    search_specific_flights,
    find_hotels,
    get_coordinates,
    get_weather,
    get_driving_distance,
]

all_tools = internal_tools + mcp_tools
print("All tools available to agent:", [t.name for t in all_tools])


NameError: name 'mcp_tools' is not defined